# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library. We follow a structured workflow to:
- Load dataset metadata
- Inspect available record sets and fields (using their `@id`)
- Extract and process records
- Perform exploratory data analysis (EDA)
- Visualize data

### Dataset Source
This dataset's Croissant schema (JSON-LD) can be accessed here:

https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and available records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.simplefilter('ignore')

# The Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print basic metadata summary
print(f"Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}")
print(f"Identifier: {dataset.metadata.identifier}\n")
print(f"Keywords: {dataset.metadata.keywords}")


## 2. Data Overview
List all available record sets (via `@id`), their fields, and column IDs included in the Croissant dataset. This provides a map for record extraction and referencing downstream fields by their `@id`.

In [ ]:
# Enumerate all record sets and their fields.
# The 'record_sets' property is a list of RecordSet objects
if hasattr(dataset, 'record_sets'):
    record_sets = dataset.record_sets
else:
    # fallback for older mlcroissant naming
    record_sets = dataset.recordsets

print("Available Record Sets (by @id):\n")
record_set_ids = []
for rset in record_sets:
    print(f"- RecordSet Name: {getattr(rset, 'name', None)}")
    print(f"  @id: {rset.id}")
    record_set_ids.append(rset.id)
    # List corresponding fields
    if hasattr(rset, 'fields'):
        print("  Fields:")
        for field in rset.fields:
            print(f"    - {field.name} (@id: {field.id})")
    # List columns if present (often mapped per field)
    if hasattr(rset, 'columns') and rset.columns:
        print("  Columns:")
        for col in rset.columns:
            print(f"    - {col.name} (@id: {col.id})")
    print()
if not record_set_ids:
    print("No record sets found. Double-check schema definition at the provided URL.")

## 3. Data Extraction
Extract the records from each available record set by their `@id` and load them into Pandas DataFrames. Downstream analysis will work off these DataFrames, referenced by the record set `@id`.

In [ ]:
# Prepare: Collect DataFrames for all available record sets
all_dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nExtracting records for RecordSet @id: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            all_dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records. Fields: {list(df.columns)}")
        else:
            print("No records found in this set.")
    except Exception as e:
        print(f"Error extracting for {record_set_id}: {e}")

if not all_dataframes:
    print("No record sets yielded records. Verify dataset availability and schema completeness.")
else:
    # Example: Display head of the first available DataFrame
    first_record_set_id = list(all_dataframes.keys())[0]
    print(f"\nExample DataFrame for RecordSet: {first_record_set_id}")
    display(all_dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's select a record set of interest (by `@id`) and perform some example EDA operations. This includes filtering on a numeric field, normalizing the field, and grouping the data. All field references will use their Croissant `@id`.

In [ ]:
# Choose record set and numeric field for demonstration

import numpy as np

if all_dataframes:
    # Use the first record set with available numeric columns
    selected_record_set_id = None
    selected_numeric_field_id = None
    selected_group_field_id = None
    
    for rid, df in all_dataframes.items():
        # Infer numeric columns
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            selected_record_set_id = rid
            selected_numeric_field_id = numeric_cols[0]
            # Try to find another field for grouping -- a string or categorical
            cand_group = [col for col in df.select_dtypes(include=[object]).columns if col != selected_numeric_field_id]
            if cand_group:
                selected_group_field_id = cand_group[0]
            break

    print(f"Selected RecordSet @id: {selected_record_set_id}")
    print(f"Selected numeric field @id: {selected_numeric_field_id}")
    if selected_group_field_id:
        print(f"Group-by field @id: {selected_group_field_id}")

    df = all_dataframes[selected_record_set_id]
    # Remove missing values for the selected numeric field
    filtered_df = df[df[selected_numeric_field_id].notnull()]

    threshold = np.nanmedian(filtered_df[selected_numeric_field_id])
    filtered_df = filtered_df[filtered_df[selected_numeric_field_id] > threshold]
    print(f"Filtered records with {selected_numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    norm_col = selected_numeric_field_id + '_normalized'
    filtered_df[norm_col] = (filtered_df[selected_numeric_field_id] -
                             filtered_df[selected_numeric_field_id].mean()) / \
                             filtered_df[selected_numeric_field_id].std()
    print(f"Normalized {selected_numeric_field_id} for filtered records:")
    display(filtered_df[[selected_numeric_field_id, norm_col]].head())

    # Group by
    if selected_group_field_id and selected_group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(selected_group_field_id)[selected_numeric_field_id].mean().reset_index()
        print(f"Mean of {selected_numeric_field_id} grouped by {selected_group_field_id}:")
        display(grouped_df.head())
else:
    print("No suitable DataFrame with numeric fields for EDA available.")

## 5. Visualization
Let's produce some basic visualizations: the distribution of our selected numeric field, and its mean by group. All axes and legends will reference columns by their Croissant `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if all_dataframes and selected_record_set_id and selected_numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[selected_numeric_field_id].dropna(), kde=True, bins=25)
    plt.title(f"Distribution of '{selected_numeric_field_id}' in RecordSet '{selected_record_set_id}'")
    plt.xlabel(selected_numeric_field_id)
    plt.show()

    if selected_group_field_id and selected_group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        order = df[selected_group_field_id].value_counts().index
        sns.boxplot(x=selected_group_field_id, y=selected_numeric_field_id, data=df, order=order)
        plt.title(f"'{selected_numeric_field_id}' by '{selected_group_field_id}' in '{selected_record_set_id}'")
        plt.xlabel(selected_group_field_id)
        plt.ylabel(selected_numeric_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()


## 6. Conclusion

In this notebook, we've demonstrated how to load, explore, and process a Croissant-formatted dataset with the `mlcroissant` library. By referencing all record sets and fields by their unique `@id`, we ensure reproducibility and schema alignment. You can extend this workflow for further statistical analysis, model training, and detailed domain-specific interpretation.

**Key findings and notes:**
- Dataset metadata and structure can be programmatically navigated via Croissant's `@id` fields.
- Data can be loaded and analyzed directly from the official schema URL, with all entities referenced explicitly.
- The EDA demonstrated filtering, normalization, and grouping, while visualizations provided a snapshot of numeric field distributions and group-wise differences.

Continue adapting this notebook and the dataset schema for your analysis needs!